# Sirius TPC-H benchmark — cold vs. lukewarm

Replicates the timing logic of `new_sirius/test/tpch_performance/performance_test.py`
(`--execution cold` and `--execution lukewarm`) against parquet on S3.

| profile | ordering | between runs | measures |
|---|---|---|---|
| **cold** | sequential over queries | `CALL reset_sirius_cache()` before *every* query | a scan that finds nothing cached |
| **lukewarm** | sequential over queries | nothing | LRU cache under real pressure — iteration *n+1* of a query only comes after every other query has run |

`echo 3 > /proc/sys/vm/drop_caches` is intentionally **not** done: the data lives on S3, so there is
no local page cache holding it. Sirius's own prefetching cache lives in pinned host memory and
survives an OS drop anyway — `reset_sirius_cache()` is the only thing that actually makes a run cold.


## 0. Refresh the S3 credentials in `sirius.yml`

The `access_key` / `secret_key` / `session_token` in the Sirius config are a **temporary** STS
session from this instance's IAM role and expire after a few hours. Once they lapse every scan
fails at bind time, so refresh before anything else — `LOAD` reads the file once and keeps what it
read, meaning a refresh after connecting does not reach the running extension.

In [ ]:
import subprocess

REPO = "/home/ubuntu/workspace/new_sirius"
CONFIG = "/home/ubuntu/workspace/sirius.yml"
REFRESH_SCRIPT = f"{REPO}/refresh_autotune_creds.sh"

# Rewrites access_key/secret_key/session_token in place from IMDSv2. `check=True`
# so an expired-and-unrefreshable credential fails here rather than as an opaque
# S3 error 20 minutes into the sweep.
proc = subprocess.run(
    ["bash", REFRESH_SCRIPT, CONFIG],
    capture_output=True, text=True, check=True,
)
print(proc.stdout.strip() or proc.stderr.strip())

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import statistics
import time
from collections.abc import Iterable, Sequence
from dataclasses import dataclass, field
from datetime import datetime

import duckdb
import pandas as pd

EXTENSION = f"{REPO}/build/release/extension/sirius/sirius.duckdb_extension"
S3_BASE = "s3://sirius-s3-test/datasets/tpch_sf1000"

TPCH_TABLES = (
    "customer", "lineitem", "nation", "orders",
    "part", "partsupp", "region", "supplier",
)

# Sirius reads SIRIUS_CONFIG_FILE at LOAD time, so it must be set before connecting --
# and after the credential refresh above. The YAML already carries the cache settings
# both profiles want (`scan_manager.cache: {mode: sirius, eviction: lru}`); cold relies
# on the explicit reset below, not on the evictor.
os.environ["SIRIUS_CONFIG_FILE"] = CONFIG

In [ ]:
def open_connection() -> duckdb.DuckDBPyConnection:
    """In-memory DuckDB with Sirius loaded and the 8 TPC-H views bound to S3.

    Autoload is disabled so an `s3://` bind cannot pull in DuckDB's own httpfs and
    quietly serve the scan on the CPU; it must resolve through `sirius_httpfs`.
    Sirius is loaded *before* the views are created — that registration is what
    makes the s3:// glob bind at all.
    """
    con = duckdb.connect(config={
        "allow_unsigned_extensions": "true",
        "autoinstall_known_extensions": "false",
        "autoload_known_extensions": "false",
    })
    con.execute(f"LOAD '{EXTENSION}'")
    for table in TPCH_TABLES:
        con.execute(
            f"CREATE OR REPLACE VIEW {table} AS "
            f"SELECT * FROM read_parquet('{S3_BASE}/{table}/*.parquet')"
        )
    con.execute("SET gpu_execution = true")
    return con


con = open_connection()

## 2. TPC-H queries

Pulled straight from DuckDB's own `tpch` extension via `tpch_queries()` — the 22 official texts,
no hand-maintained copy to drift. They are fetched on a **throwaway connection**: the benchmark
connection has `autoload_known_extensions=false` (so an `s3://` bind cannot fall back to DuckDB's
httpfs), and we do not want the `tpch` extension sitting in the catalog being measured.

All 22 are single statements — DuckDB writes q15 with a CTE rather than the spec's
`CREATE VIEW revenue0`, so each one runs in a single `execute()`.

In [ ]:
SCALE_FACTOR = 1000   # tpch_sf1000; used only to scale q11's threshold


def load_tpch_queries(scale_factor: float = 1) -> dict[str, str]:
    """The 22 official TPC-H texts from DuckDB's tpch extension, keyed q1..q22.

    q11's HAVING threshold is the spec's `0.0001 / SF`, but tpch_queries() emits
    the SF1 constant, so at SF1000 it sits ~1000x too high and the query returns
    zero rows. Runtime is unaffected (the HAVING prunes only after the scan, join
    and aggregate are done) -- rescale it purely so the result looks sane. Delete
    the fixup if you would rather compare against the unscaled text.
    """
    with duckdb.connect() as scratch:
        scratch.execute("INSTALL tpch; LOAD tpch")
        rows = scratch.execute(
            "SELECT query_nr, query FROM tpch_queries() ORDER BY query_nr"
        ).fetchall()

    queries = {f"q{nr}": sql.strip().rstrip(";") for nr, sql in rows}
    queries["q11"] = queries["q11"].replace(
        "* 0.0001000000", f"* {0.0001 / scale_factor:.10f}"
    )
    return queries


TPCH_QUERIES = load_tpch_queries(SCALE_FACTOR)
print(f"{len(TPCH_QUERIES)} queries: {', '.join(TPCH_QUERIES)}")

## 3. Timing harness

In [ ]:
@dataclass(slots=True)
class Timing:
    query: str
    iteration: int
    profile: str
    seconds: float
    rows: int


def time_query(con: duckdb.DuckDBPyConnection, name: str) -> tuple[float, int]:
    """Wall-clock a single query, fetch included (that is what the script measures)."""
    start = time.perf_counter()
    rows = con.execute(TPCH_QUERIES[name]).fetchall()
    return time.perf_counter() - start, len(rows)


def reset_sirius_cache(con: duckdb.DuckDBPyConnection) -> None:
    """Drop Sirius's prefetching cache. This is what makes a run cold."""
    con.execute("CALL reset_sirius_cache();").fetchall()


def run_profile(
    con: duckdb.DuckDBPyConnection,
    profile: str,
    *,
    queries: Sequence[str],
    iterations: int = 3,
    reset_between: bool,
    verbose: bool = True,
) -> list[Timing]:
    """Round-robin ('sequential' ordering): all queries per iteration, iterations outermost."""
    results: list[Timing] = []
    for it in range(iterations):
        for name in queries:
            if reset_between:
                reset_sirius_cache(con)
            try:
                secs, nrows = time_query(con, name)
            except Exception as exc:  # a failure must not poison the rest of the sweep
                secs, nrows = float("nan"), -1
                if verbose:
                    print(f"  {name} iter{it}: FAILED — {type(exc).__name__}: {exc}")
            if verbose and nrows >= 0:
                ts = datetime.now().strftime("%H:%M:%S")
                print(f"[{ts}] {profile:>9} | {name:>3} iter{it} | {secs:8.4f}s | {nrows} rows")
            results.append(Timing(name, it, profile, secs, nrows))
    return results


def summarize(timings: Iterable[Timing]) -> pd.DataFrame:
    df = pd.DataFrame(timings)
    wide = (
        df.pivot_table(index=["profile", "query"], columns="iteration", values="seconds")
          .add_prefix("iter")
    )
    wide["mean"] = wide.mean(axis=1)   # over the iterations, before `best` is appended
    wide["best"] = wide.filter(like="iter").min(axis=1)
    order = {q: i for i, q in enumerate(TPCH_QUERIES)}
    return wide.sort_index(key=lambda idx: idx.map(order) if idx.name == "query" else idx)


QUERIES_TO_RUN = list(TPCH_QUERIES)   # or e.g. ["q1", "q5", "q9"]
ITERATIONS = 3
all_timings: list[Timing] = []

## 4. Cold

`reset_sirius_cache()` before **every** query, so nothing is carried over from the previous run.
Round-robin ordering. The connection is deliberately *not* reopened between runs — that would also
throw away the GPU context and the compiled plans, which is not what this is measuring.

In [ ]:
cold = run_profile(
    con,
    "cold",
    queries=QUERIES_TO_RUN,
    iterations=1,
    reset_between=True,
)
all_timings += cold
summarize(cold)

## 5. Lukewarm

No cache reset at all — the cache is left to fill and evict under its own LRU policy. Round-robin
ordering matters here: a query's second iteration lands after all 21 others have run, so it finds a
cache under genuine pressure rather than its own leftovers.

Run this **after** the cold cell in the same session, exactly as the script does.

In [ ]:
lukewarm = run_profile(
    con,
    "lukewarm",
    queries=QUERIES_TO_RUN,
    iterations=1,
    reset_between=False,     # <- the only difference from cold
)
all_timings += lukewarm
summarize(lukewarm)

## 6. Side by side

In [ ]:
summary = summarize(all_timings)
best = summary["best"].unstack("profile")
best["speedup"] = best["cold"] / best["lukewarm"]
totals = best[["cold", "lukewarm"]].sum()

print(f"TOTAL cold     : {totals['cold']:.3f}s")
print(f"TOTAL lukewarm : {totals['lukewarm']:.3f}s  ({totals['cold'] / totals['lukewarm']:.2f}x)")
best.style.format({"cold": "{:.4f}", "lukewarm": "{:.4f}", "speedup": "{:.2f}x"})

In [ ]:
pd.DataFrame(all_timings).to_csv("tpch_runtimes.csv", index=False)
con.close()